# 01 · YouTube → caption text

Download a YouTube video, transcribe with [WhisperX](https://github.com/m-bain/whisperx)
(Whisper large-v2 + wav2vec2 alignment), and write a cleaned `.txt` of the spoken content.

**Heads up — WhisperX needs torch 1.13.1**, which conflicts with the classifier's
PyTorch ≥ 2.0. The whisper toolchain therefore lives in its own virtualenv
(`whisper-env`), set up in cell 1. After that, the python pipeline in `src/extract_subtitles.py`
calls the binary as a subprocess.

## 1. Bootstrap the WhisperX virtualenv (one-time)

In [ ]:
# Build whisper-env once per machine. Skip this cell once it's set up.
!pip install -q virtualenv
!virtualenv -q whisper-env
!whisper-env/bin/pip install -q --index-url https://download.pytorch.org/whl/cu117 torch==1.13.1 torchaudio==0.13.1
!whisper-env/bin/pip install -q git+https://github.com/m-bain/whisperx.git@v2.0.1

## 2. Make `src/` importable

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["WHISPERX_BIN"] = str(ROOT / "whisper-env" / "bin" / "whisperx")

## 3. Run the pipeline

Output goes to `output/`: `audio.mp3` → `audio.srt` → `audio.txt`.

In [ ]:
from src.extract_subtitles import extract

URL = "https://www.youtube.com/watch?v=4sC-k-92JBE"  # @param {type:"string"}
OUT_DIR = ROOT / "output"

txt_path = extract(URL, OUT_DIR, language="en", model="large-v2")
print(f"wrote: {txt_path}")

## 4. Preview

In [ ]:
print(txt_path.read_text(encoding="utf-8")[:600])